# 06 New Findings: testing the week 7 next steps

06 explained where the error comes from. This notebook tests the three changes it pointed to, against 05's best model.

| ID | Change | Why, from 06 |
|---|---|---|
| W0 | Reference: random forest on geo + columns + HOA (`kept_groups`) | Best 05 model, 7.50% CV MdAPE |
| W1 | W0 + a month index | March was underpredicted by 1.1% every week (E8). The model has no sense of time beyond season |
| W2 | Gradient boosting on the same features | Forests average leaves, which pulls predictions toward the middle (E7) |
| W3 | W2 + month index + similar comps | Boosting can use a strong single feature better than a forest that samples it in some trees only |
| W4 | Comps as the starting point, random forest | Similar comps alone reach 9.5% MdAPE (E4). Model log(price ÷ comp value) instead of the price |
| W5 | Comps as the starting point, gradient boosting | Same as W4 with boosting |

**How every change is judged.** Rolling time-based CV over Nov 2025 to Mar 2026 (train on every earlier month). A change **helps** only if the paired bootstrap 95% interval of the MdAPE change is below 0 **and** at least 4 of 5 months are better. Same rule as 05 and 06.

**Rules.** ListPrice, OriginalListPrice, DaysOnMarket and contract dates are never features (leakage, `plan.md` §04). The test month (Apr 2026) is never read.

In [1]:
import os
import time

import numpy as np
import pandas as pd
import sklearn
from sklearn.cluster import MiniBatchKMeans
from sklearn.compose import TransformedTargetRegressor
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.pipeline import Pipeline

from idx_pipeline import (CAT_COLS, DATA_DIR, SEED, TARGET, DatasetAImputer, add_basic_features, add_extras,
                          add_fee_features, add_geo_features, add_similar_comps, build_comp_pool, build_preprocessor,
                          fill_coords, load_extras, load_split, metrics, rolling_folds)

print("sklearn", sklearn.__version__)
np.random.seed(SEED)
MODEL_DIR = os.path.join(DATA_DIR, "models")
VAL_MONTHS = ["2025-11", "2025-12", "2026-01", "2026-02", "2026-03"]

description = pd.read_csv(os.path.join(DATA_DIR, "dataset_description.csv"))
features = description.loc[description["role"].isin(["model_feature", "missing_flag"]), "column"].tolist()
BASE_NUM = [c for c in features if c not in CAT_COLS]  # Dataset A, same as 04 and 05

# 05's kept groups (columns + HOA + geo), copied from 05 so the reference is the same model
KEPT_NUM = BASE_NUM + [
    "bed_bath_ratio", "sqft_per_bed", "baths_per_bed", "stories_filled", "yard_sqft", "lot_coverage",
    "main_level_share", "has_wood", "has_stone", "has_tile", "has_carpet", "has_laminate", "has_vinyl",
    "Flooring_missing", "in_subdivision", "flag_small_for_beds", "flag_many_baths",
    "fee_monthly", "HasHOA", "AssociationFee_missing",
    "rot_15", "rot_30", "rot_45", "rot_60", "km_la", "km_sf", "km_sj", "km_sd", "km_sac", "km_job_center", "km_coast"]
KEPT_CAT = CAT_COLS + ["MLSAreaMajor", "SubdivisionName", "geo_cluster"]
SIM_COMPS = ["sim_ppsf", "sim_ppsf_iqr", "sim_count", "sim_value"]
TIME = ["month_idx"]

LEAKY = ["ListPrice", "DaysOnMarket", "ContractDate", "OnMarketDate", "PendingTimestamp"]
for col in KEPT_NUM + KEPT_CAT + SIM_COMPS + TIME:
    assert not any(word.lower() in col.lower() for word in LEAKY), f"leakage column {col}"
print(len(KEPT_NUM), "numeric,", len(KEPT_CAT), "categorical in the reference")

sklearn 1.9.1
62 numeric, 6 categorical in the reference


### Step 1: Folds
Same five folds and the same features as 05, plus similar comps (06) and a month index. Everything learned from data (the imputer, the 100 geo clusters, the comp pool) is fit on each fold's train rows only. The month index is just months since the first month in the data, from `CloseDate`, so nothing is learned.

In [2]:
dev = pd.concat([load_split(s) for s in ["train", "val"]], ignore_index=True)  # test stays out
extras = load_extras()  # raw columns Dataset A leaves out, read up to Mar 2026 only
first = dev["CloseDate"].min().to_period("M")
print("first month", first, "rows", len(dev))


def engineer(df, lat, lon, pool, kmeans):
    out = add_fee_features(add_basic_features(add_extras(df, extras)))
    out = add_geo_features(out, lat, lon)
    out["geo_cluster"] = kmeans.predict(np.column_stack([lat, lon])).astype(str)
    out = add_similar_comps(out, lat, lon, pool)
    out["month_idx"] = (out["CloseDate"].dt.to_period("M") - first).apply(lambda d: d.n).astype(float)
    return out


folds = {}
start = time.time()
for month, train, val in rolling_folds(dev, VAL_MONTHS):
    assert train["source_month"].max() < month  # train is always earlier than validation
    imputer = DatasetAImputer().fit(train)
    lat, lon = fill_coords(train, imputer)
    pool = build_comp_pool(train, lat, lon)
    kmeans = MiniBatchKMeans(n_clusters=100, n_init=3, random_state=SEED).fit(np.column_stack([lat, lon]))
    folds[month] = (engineer(train, lat, lon, pool, kmeans), engineer(val, *fill_coords(val, imputer), pool, kmeans))
    print(month, "train", len(train), "val", len(val), f"{time.time() - start:.0f}s")

val_all = pd.concat([folds[m][1] for m in VAL_MONTHS], ignore_index=True)  # every validation home, in order
y_all = val_all[TARGET].to_numpy()

first month 2024-01 rows 293691
2025-11 train 246137 val 9596 45s
2025-12 train 255733 val 10317 98s
2026-01 train 266050 val 7370 149s
2026-02 train 273420 val 8859 202s
2026-03 train 282279 val 11412 274s


### Step 2: CV helpers
`run` fits one model per fold and scores the pooled validation homes. In **comp baseline** mode the model learns log(price ÷ similar-comp value), and the prediction is the comp value times the learned adjustment. Homes with no similar comps fall back to another model's prediction, so every config is scored on exactly the same homes.

Gradient boosting is sklearn's `HistGradientBoostingRegressor` (the same family as LightGBM, which isn't installed). Early stopping uses a random 10% of each fold's **train** rows, never the validation month.

In [3]:
def rf(num, cat):
    return Pipeline([("pre", build_preprocessor("tree_te", num, cat)),
                     ("model", RandomForestRegressor(n_estimators=100, min_samples_leaf=5, n_jobs=-1, random_state=SEED))])


def hgb(num, cat):
    return Pipeline([("pre", build_preprocessor("tree_te", num, cat)),
                     ("model", HistGradientBoostingRegressor(learning_rate=0.05, max_iter=2000, max_leaf_nodes=63,
                                                             min_samples_leaf=20, l2_regularization=1.0,
                                                             early_stopping=True, validation_fraction=0.1,
                                                             n_iter_no_change=50, random_state=SEED))])


rows, preds, covered = {}, {}, {}


def has_comp(df):
    return (df["sim_value"] > 0).to_numpy()  # NaN compares False


def run(name, make, num, cat, comp_base=False, fallback=None):
    num, cat = list(num), list(cat)
    cols = list(dict.fromkeys(num + cat))
    start, out, per_fold, cov = time.time(), [], {}, []
    for month in VAL_MONTHS:
        train, val = folds[month]
        if comp_base:
            t = has_comp(train)
            model = make(num, cat).fit(train.loc[t, cols], np.log(train.loc[t, TARGET] / train.loc[t, "sim_value"]))
            v = has_comp(val)
            p = preds[fallback][month].copy()
            p[v] = val.loc[v, "sim_value"].to_numpy() * np.exp(model.predict(val.loc[v, cols]))
            cov.append(v)
        else:
            model = TransformedTargetRegressor(regressor=make(num, cat), func=np.log, inverse_func=np.exp)
            model.fit(train[cols], train[TARGET].to_numpy())
            p = model.predict(val[cols])
        out.append(p)
        per_fold[f"MdAPE_{month}"] = metrics(val[TARGET].to_numpy(), p)["MdAPE_%"]
    # overfit gap on the last fold only, like 05
    if comp_base:
        t = has_comp(train)
        tp = train.loc[t, "sim_value"].to_numpy() * np.exp(model.predict(train.loc[t, cols]))
        train_score = metrics(train.loc[t, TARGET].to_numpy(), tp)["MdAPE_%"]
    else:
        train_score = metrics(train[TARGET].to_numpy(), model.predict(train[cols]))["MdAPE_%"]
    preds[name] = dict(zip(VAL_MONTHS, out))
    pred = np.concatenate(out)
    rows[name] = {**{f"cv_{k}": v for k, v in metrics(y_all, pred).items()}, **per_fold,
                  "gap_last_fold": round(per_fold[f"MdAPE_{VAL_MONTHS[-1]}"] - train_score, 2),
                  "n_inputs": len(cols), "fit_s": round(time.time() - start)}
    if comp_base:
        covered[name] = np.concatenate(cov)
    print(name, rows[name]["cv_MdAPE_%"], rows[name]["cv_MAPE_%"], f"{rows[name]['fit_s']}s")


def pooled(name):
    return np.concatenate([preds[name][m] for m in VAL_MONTHS])


def bootstrap_change(new, ref="W0", n=1000):
    # paired: the same resampled homes for both models. Negative means the new model is better.
    a_all, b_all = np.abs(pooled(new) - y_all) / y_all, np.abs(pooled(ref) - y_all) / y_all
    rng = np.random.default_rng(SEED)
    idx = rng.integers(0, len(y_all), (n, len(y_all)))
    a, b = a_all[idx], b_all[idx]
    med = 100 * (np.median(a, axis=1) - np.median(b, axis=1))
    mean = 100 * (a.mean(axis=1) - b.mean(axis=1))
    wins = sum(rows[new][f"MdAPE_{m}"] < rows[ref][f"MdAPE_{m}"] for m in VAL_MONTHS)
    return {"MdAPE_change": round(100 * (np.median(a_all) - np.median(b_all)), 2),
            "ci_low": round(np.percentile(med, 2.5), 2), "ci_high": round(np.percentile(med, 97.5), 2),
            "folds_better": wins,
            "MAPE_change": round(100 * (a_all.mean() - b_all.mean()), 2),
            "mape_ci_low": round(np.percentile(mean, 2.5), 2), "mape_ci_high": round(np.percentile(mean, 97.5), 2)}

### Step 3: Reference (W0)
This should reproduce 05's `kept_groups`: 7.50% MdAPE and 11.22% MAPE in `fe_cv_metrics.csv`.

In [4]:
run("W0", rf, KEPT_NUM, KEPT_CAT)
ref05 = pd.read_csv(os.path.join(MODEL_DIR, "fe_cv_metrics.csv"), index_col=0).loc["kept_groups"]
print("05 kept_groups:", ref05["cv_MdAPE_%"], ref05["cv_MAPE_%"], "| W0:", rows["W0"]["cv_MdAPE_%"], rows["W0"]["cv_MAPE_%"])

W0 7.5 11.22 788s
05 kept_groups: 7.5 11.22 | W0: 7.5 11.22


### Step 4: Time index and gradient boosting (W1 to W3)

In [5]:
run("W1", rf, KEPT_NUM + TIME, KEPT_CAT)
run("W2", hgb, KEPT_NUM, KEPT_CAT)
run("W3", hgb, KEPT_NUM + TIME + SIM_COMPS, KEPT_CAT)

W1 7.44 11.2 653s
W2 7.24 10.45 605s
W3 7.12 10.34 477s


### Step 5: Comps as the starting point (W4, W5)
The model only has to learn how this home differs from its similar recent sales. Homes without similar comps (missing size or year, or no similar sale nearby in 180 days) fall back to W0 for the forest and W2 for boosting.

In [6]:
run("W4", rf, KEPT_NUM + TIME + SIM_COMPS, KEPT_CAT, comp_base=True, fallback="W0")
run("W5", hgb, KEPT_NUM + TIME + SIM_COMPS, KEPT_CAT, comp_base=True, fallback="W2")

cov = covered["W5"]
print(f"homes with similar comps: {cov.mean():.1%}")
split = {}
for name in ["W0", "W2", "W4", "W5"]:
    p = pooled(name)
    split[name] = {"MdAPE_with_comps": metrics(y_all[cov], p[cov])["MdAPE_%"],
                   "MdAPE_without_comps": metrics(y_all[~cov], p[~cov])["MdAPE_%"]}
split["comps_alone"] = {"MdAPE_with_comps": metrics(y_all[cov], val_all["sim_value"].to_numpy()[cov])["MdAPE_%"],
                        "MdAPE_without_comps": np.nan}
pd.DataFrame(split).T

W4 8.07 11.84 867s
W5 7.21 10.51 472s
homes with similar comps: 97.7%


,MdAPE_with_comps,MdAPE_without_comps
W0,7.40,14.20
W2,7.16,12.02
W4,7.98,14.20
W5,7.12,12.02
comps_alone,9.52,NaN


### Step 6: Results

In [7]:
NAMES = {"W0": "RF kept_groups (reference)", "W1": "RF + month index", "W2": "Gradient boosting",
         "W3": "GB + month index + similar comps", "W4": "Comp baseline + RF", "W5": "Comp baseline + GB"}
cv = pd.DataFrame(rows).T
cv.insert(0, "config", pd.Series(NAMES))
cv.to_csv(os.path.join(MODEL_DIR, "06_new_findings_cv.csv"))
cv

,config,cv_MdAPE_%,cv_MAPE_%,cv_R2,cv_MAE_$,cv_within_10%,MdAPE_2025-11,MdAPE_2025-12,MdAPE_2026-01,MdAPE_2026-02,MdAPE_2026-03,gap_last_fold,n_inputs,fit_s
W0,RF kept_groups (reference),7.50,11.22,0.8911,147016.0,61.2,7.47,7.44,7.54,7.65,7.46,2.87,68.0,788.0
W1,RF + month index,7.44,11.20,0.8913,146826.0,61.3,7.39,7.40,7.45,7.56,7.42,2.90,69.0,653.0
W2,Gradient boosting,7.24,10.45,0.9119,134931.0,63.1,7.22,7.18,7.37,7.42,7.08,0.95,68.0,605.0
W3,GB + month index + similar comps,7.12,10.34,0.9110,134768.0,63.6,7.06,7.04,7.21,7.33,7.04,0.98,73.0,477.0
W4,Comp baseline + RF,8.07,11.84,0.8845,154152.0,58.6,7.75,7.98,8.32,8.35,8.11,3.35,73.0,867.0
W5,Comp baseline + GB,7.21,10.51,0.9097,136290.0,63.2,7.10,7.17,7.26,7.42,7.13,1.19,73.0,472.0


In [8]:
vs_ref = pd.DataFrame({n: bootstrap_change(n) for n in rows if n != "W0"}).T
vs_ref.insert(0, "config", pd.Series(NAMES))
vs_ref["helps"] = (vs_ref["ci_high"] < 0) & (vs_ref["folds_better"] >= 4)
vs_ref["hurts"] = vs_ref["ci_low"] > 0
vs_ref.to_csv(os.path.join(MODEL_DIR, "06_new_findings_vs_ref.csv"))
vs_ref.sort_values("MdAPE_change")

,config,MdAPE_change,ci_low,ci_high,folds_better,MAPE_change,mape_ci_low,mape_ci_high,helps,hurts
W3,GB + month index + similar comps,-0.38,-0.46,-0.29,5.0,-0.88,-0.95,-0.82,True,False
W5,Comp baseline + GB,-0.29,-0.38,-0.20,5.0,-0.71,-0.78,-0.64,True,False
W2,Gradient boosting,-0.26,-0.34,-0.18,5.0,-0.77,-0.84,-0.71,True,False
W1,RF + month index,-0.05,-0.09,-0.02,5.0,-0.02,-0.03,-0.01,True,False
W4,Comp baseline + RF,0.57,0.48,0.66,0.0,0.62,0.54,0.70,False,True


### Step 7: Did the biases from 06 shrink?
Signed bias is the median of (prediction − actual) ÷ actual. Positive means overpredicted. Compared for the reference and the config with the lowest CV MdAPE.

In [9]:
best = cv["cv_MdAPE_%"].astype(float).idxmin()
print("best config:", best, NAMES[best])


def bias_table(groups=None):
    # groups=None means deciles of each model's own prediction
    out = {}
    for name in dict.fromkeys(["W0", best]):
        p = pooled(name)
        g = pd.qcut(p, 10, labels=range(1, 11)) if groups is None else groups
        d = pd.DataFrame({"g": g, "err": (p - y_all) / y_all})
        out[(name, "bias_%")] = d.groupby("g", observed=True)["err"].median() * 100
        out[(name, "MdAPE_%")] = d.assign(err=d["err"].abs()).groupby("g", observed=True)["err"].median() * 100
    t = pd.DataFrame(out).round(2)
    if groups is not None:
        t["n"] = pd.Series(groups).value_counts()
    return t


print("By actual price decile (1 = cheapest)")
display(bias_table(pd.qcut(y_all, 10, labels=range(1, 11))))
print("By predicted price decile, each model grouped by its own prediction")
display(bias_table())

best config: W3 GB + month index + similar comps
By actual price decile (1 = cheapest)


W0             W3             n
   bias_% MdAPE_% bias_% MdAPE_%      
g                                     
1    4.45    8.12   3.32    7.84  4788
2    0.67    6.04   0.43    5.81  4859
3    0.35    5.84  -0.22    5.65  4728
4    0.62    5.64  -0.21    5.54  4810
5   -0.00    5.77  -0.51    5.81  4601
6   -0.06    6.82  -0.26    6.55  4746
7   -0.84    8.30  -1.29    7.64  4764
8   -1.74    8.60  -1.83    8.09  4878
9   -2.90    9.66  -3.15    9.25  4699
10  -8.41   12.92  -6.46   11.84  4681

By predicted price decile, each model grouped by its own prediction


W0             W3        
   bias_% MdAPE_% bias_% MdAPE_%
g                               
1    0.01    7.25  -0.68    7.23
2   -0.57    5.76  -0.94    5.72
3   -0.64    5.78  -1.06    5.61
4   -0.39    5.70  -1.17    5.74
5   -0.04    5.77  -0.42    5.70
6    0.54    7.33   0.23    6.86
7   -0.40    8.99  -0.72    8.15
8   -0.06    8.80  -0.52    8.26
9   -0.10   10.16  -0.97    9.45
10  -1.56   12.32  -0.52   11.42

In [10]:
print("By validation month (E8 was the March trend bias)")
display(bias_table(val_all["source_month"].to_numpy()))
top = val_all["CountyOrParish"].value_counts().index[:10]
county = val_all["CountyOrParish"].where(val_all["CountyOrParish"].isin(top), "Other").to_numpy()
print("By county, 10 largest")
display(bias_table(county).sort_values(("W0", "MdAPE_%")))

By validation month (E8 was the March trend bias)


W0             W3              n
        bias_% MdAPE_% bias_% MdAPE_%       
g                                           
2025-11   0.22    7.47  -0.32    7.06   9596
2025-12   0.47    7.44  -0.33    7.04  10317
2026-01   0.16    7.54  -0.63    7.21   7370
2026-02  -0.81    7.65  -1.35    7.33   8859
2026-03  -1.32    7.46  -0.92    7.04  11412

By county, 10 largest


W0             W3              n
                bias_% MdAPE_% bias_% MdAPE_%       
g                                                   
Riverside        -0.12    5.53  -0.40    5.52   7674
San Bernardino   -0.88    6.26  -1.08    6.25   5343
Ventura           0.38    6.53  -0.00    6.74   1542
Contra Costa      2.02    7.19  -0.88    7.23   2150
Orange           -2.04    7.56  -0.89    7.09   4502
San Diego        -0.49    7.58  -1.15    7.34   5260
Other             0.26    8.55  -0.67    7.92   5132
Los Angeles      -0.24    8.61  -0.50    7.88  11818
Santa Clara      -0.68    9.03  -1.09    8.30   1233
San Luis Obispo  -0.81    9.50  -0.27    8.59    823
Alameda           1.37   10.10  -1.27    9.08   2077